# Device enforcement: brief exploratory analysis

Explore one completed Raw snapshot to inform Bronze and Silver design. Run cells
in order with the project's notebook environment. All analysis stays in memory;
source JSON is preserved, and no downloads or dataset writes are performed.

## 1. Load one snapshot

The relative batch path below explicitly selects the verified initial download.
Change it to inspect another completed batch; never combine snapshots. Root
resolution supports launching from the repository root or `notebooks/`.

In [ ]:
from collections import Counter
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

repo_root = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "pyproject.toml").is_file() and (path / "docs/PIPELINE.md").is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Open this notebook from the repository root or notebooks/.")

batch_relative_path = Path(
    "data/raw/device_enforcement/ingestion_date=2026-10-23/"
    "batch_id=20261023T041846804814Z_2cd803ad618f4b798c5721e10393674d"
)
batch_dir = repo_root / batch_relative_path
metadata_path = batch_dir / "ingestion_metadata.json"
if not metadata_path.is_file():
    raise FileNotFoundError("Select a completed Raw batch containing ingestion_metadata.json.")
metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
assert metadata["endpoint"] == "/device/enforcement", "Select a device enforcement batch."
assert batch_dir.name == f"batch_id={metadata['batch_id']}", "Batch identity mismatch."
pd.set_option("display.max_colwidth", 80)

In [ ]:
records = []
file_summary = []
for partition in metadata["partitions"]:
    partition_count = 0
    for file_info in partition["files"]:
        source_path = batch_dir / file_info["filename"]
        with source_path.open("rb") as source:
            payload = json.load(source)
        file_records = payload["results"]
        assert isinstance(file_records, list), f"Expected results list: {source_path.name}"
        assert len(file_records) == file_info["actual_records"], source_path.name
        records.extend(file_records)
        partition_count += len(file_records)
        file_summary.append({"file": source_path.name, "records": len(file_records)})
        del payload, file_records
    assert partition_count == partition["expected_records"] == partition["actual_records"]
assert len(records) == metadata["expected_records"] == metadata["actual_records"]

df = pd.DataFrame.from_records(records)
display(pd.Series({
    "export_date": metadata["export_date"],
    "batch_id": metadata["batch_id"],
    "records": len(df),
    "fields": len(df.columns),
}, name="Selected snapshot"))
display(pd.DataFrame(file_summary))

## 2. Understand the structure

Inspect original values and Python types before choosing a Parquet schema.
Nested dictionaries and lists remain intact. Check whether `openfda` contains
usable annotations before considering any flattening.

In [ ]:
display(df.head(3))
field_profiles = []
for field in sorted(df.columns):
    values = [record[field] for record in records if field in record]
    types = Counter(type(value).__name__ for value in values)
    field_profiles.append({
        "field": field,
        "pandas_dtype": str(df[field].dtype),
        "observed_python_types": dict(types),
        "absent": len(records) - len(values),
        "null": sum(value is None for value in values),
        "empty_string": sum(isinstance(value, str) and not value.strip() for value in values),
        "empty_container": sum(isinstance(value, (dict, list)) and not value for value in values),
        "nested_values": sum(isinstance(value, (dict, list)) for value in values),
    })
field_profile = pd.DataFrame(field_profiles).set_index("field")
display(field_profile[["pandas_dtype", "observed_python_types", "nested_values"]])

openfda_values = [record.get("openfda") for record in records]
openfda_populated = sum(isinstance(value, dict) and bool(value) for value in openfda_values)
openfda_keys = sorted({key for value in openfda_values if isinstance(value, dict) for key in value})
display(pd.Series({
    "populated_openfda_records": openfda_populated,
    "openfda_keys": openfda_keys,
}, name="Annotation availability"))

## 3. Check completeness and identifiers

Absent keys, explicit JSON nulls, blank strings, and empty containers are counted
separately. Identifier trimming below creates analysis-only columns. Test
`recall_number` as a product-record identifier and examine the grouping provided
by `event_id`; do not assume one row equals one recall event.

In [ ]:
missingness = field_profile[["absent", "null", "empty_string", "empty_container"]].copy()
missingness["total_empty_or_missing"] = missingness.sum(axis=1)
missingness["percent_empty_or_missing"] = 100 * missingness["total_empty_or_missing"] / len(df)
display(missingness.sort_values("total_empty_or_missing", ascending=False))

identifiers = pd.DataFrame({
    field: df[field].astype("string").str.strip().replace("", pd.NA)
    for field in ["recall_number", "event_id"]
})
recall_numbers = identifiers["recall_number"]
event_ids = identifiers["event_id"]
duplicate_recall_rows = recall_numbers.notna() & recall_numbers.duplicated(keep=False)
display(pd.Series({
    "product_records": len(df),
    "missing_recall_numbers": int(recall_numbers.isna().sum()),
    "unique_nonmissing_recall_numbers": recall_numbers.nunique(),
    "rows_with_duplicate_recall_numbers": int(duplicate_recall_rows.sum()),
    "missing_event_ids": int(event_ids.isna().sum()),
    "unique_nonmissing_event_ids": event_ids.nunique(),
}, name="Identifier checks"))
display(df.loc[recall_numbers.isna() | duplicate_recall_rows,
               ["recall_number", "event_id", "report_date", "classification"]].head(10))

In [ ]:
event_groups = identifiers.dropna(subset=["event_id"]).groupby("event_id").agg(
    product_records=("recall_number", "size"),
    distinct_recall_numbers=("recall_number", "nunique"),
)
display(event_groups["product_records"].describe().rename("Records per event"))
print("Events with multiple product records:", int((event_groups["product_records"] > 1).sum()))
display(event_groups.sort_values("product_records", ascending=False).head(10))

## 4. Inspect dates

Parse the source's `YYYYMMDD` values into a separate analysis table. Missing and
unparseable values are reported separately. Dates before 2004 are review
candidates relative to the endpoint's stated coverage, not automatically invalid
recalls. Examine negative intervals without altering the original values.

In [ ]:
date_fields = [
    "report_date", "recall_initiation_date", "center_classification_date", "termination_date",
]
parsed_dates = pd.DataFrame(index=df.index)
date_profiles = []
for field in date_fields:
    raw_dates = df[field].astype("string").str.strip().replace("", pd.NA)
    parsed_dates[field] = pd.to_datetime(raw_dates, format="%Y%m%d", errors="coerce", exact=True)
    failed = raw_dates.notna() & parsed_dates[field].isna()
    date_profiles.append({
        "field": field,
        "earliest": parsed_dates[field].min(),
        "latest": parsed_dates[field].max(),
        "missing": int(raw_dates.isna().sum()),
        "parsing_failures": int(failed.sum()),
    })
    if failed.any():
        print(f"Unparseable {field} values:")
        display(df.loc[failed, ["recall_number", field]].head(5))
date_profile = pd.DataFrame(date_profiles).set_index("field")
display(date_profile)

In [ ]:
old_initiations = parsed_dates["recall_initiation_date"] < pd.Timestamp("2004-01-01")
report_delay_days = (
    parsed_dates["report_date"] - parsed_dates["recall_initiation_date"]
).dt.days
negative_intervals = report_delay_days < 0
print("Initiation dates before 2004:", int(old_initiations.sum()))
print("Report dates before initiation dates:", int(negative_intervals.sum()))
review_columns = ["recall_number", "event_id", "recall_initiation_date", "report_date"]
display(df.loc[old_initiations | negative_intervals, review_columns].assign(
    report_delay_days=report_delay_days[old_initiations | negative_intervals],
).head(10))

## 5. Explore distributions

Counts describe records published in this snapshot, not device failure rates.
An event can appear in multiple report years, so annual unique-event counts
should not be summed to obtain the snapshot's distinct-event total. Partial
boundary years are not comparable to complete years without further analysis.
Manufacturer counts use exact source names without standardization.

In [ ]:
annual_records = pd.DataFrame({
    "report_year": parsed_dates["report_date"].dt.year,
    "event_id": event_ids,
}).dropna(subset=["report_year"])
annual_counts = annual_records.groupby("report_year").agg(
    product_records=("event_id", "size"),
    unique_events=("event_id", "nunique"),
)
annual_counts.index = annual_counts.index.astype(int)
classification_counts = df["classification"].astype("string").fillna("Missing").value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
annual_counts.plot.bar(ax=axes[0])
axes[0].set(title="Activity by report year", xlabel="Report year", ylabel="Count")
classification_counts.plot.bar(ax=axes[1], color="steelblue")
axes[1].set(title="Product records by recall classification", xlabel="Classification", ylabel="Records")
axes[1].tick_params(axis="x", rotation=30)
fig.tight_layout()
plt.show()
display(annual_counts)

In [ ]:
firm_counts = df["recalling_firm"].value_counts().head(10)
firm_counts.index.name = "Source firm name"
display(firm_counts.rename("product_records").to_frame())

## 6. Summarize findings

The table below updates for the selected batch. For the initial verified export,
expect 40,300 product records, 15,107 distinct events, report dates in 2012–2026,
one blank recall number, and no populated `openfda` annotations. These are
snapshot observations, not assumptions about every future export.

In [ ]:
display(pd.Series({
    "export_date": metadata["export_date"],
    "product_records": len(df),
    "unique_events": event_ids.nunique(),
    "report_date_min": parsed_dates["report_date"].min(),
    "report_date_max": parsed_dates["report_date"].max(),
    "missing_recall_numbers": int(recall_numbers.isna().sum()),
    "duplicate_recall_number_rows": int(duplicate_recall_rows.sum()),
    "populated_openfda_records": openfda_populated,
    "initiation_dates_before_2004": int(old_initiations.sum()),
    "negative_report_intervals": int(negative_intervals.sum()),
    "date_parsing_failures": int(date_profile["parsing_failures"].sum()),
}, name="Snapshot findings"))

**Proposed downstream policies**

- **Bronze:** retain source values and nested structures, keep identifiers as
  strings, and link records to the Raw file and batch. Do not apply the notebook's
  temporary parsing or identifier trimming during format conversion.
- **Silver identifiers:** investigate missing or duplicate `recall_number` values
  before adopting it as a product-record key. Use `event_id` for grouping, not as
  a unique row key. Define version precedence across snapshots before deduplication.
- **Silver types and quality:** parse dates with explicit missing/invalid flags;
  flag anomalous dates and negative intervals for review rather than correcting
  or dropping them automatically. Keep `product_quantity` and `code_info` as text
  until their formats are understood. Preserve unfamiliar classification values
  and distinguish recall classification from device regulatory class.
- **Limitations:** this snapshot is not a revision history or a source for recall
  lifecycle tracking. Empty `openfda` dictionaries provide no annotation-based
  device categories. Exact firm names may split the same manufacturer across
  spellings, and record counts do not measure recall risk without exposure data.

These are recommendations for later pipeline work; this notebook performs no
production transformations. Clear all outputs before committing it.